In [1]:
import formulallm.formula as f

import io
import contextlib
import os
import uuid

from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent, FunctionCallingAgentWorker
from llama_index.llms.ollama import Ollama
from llama_index.core import PromptTemplate

In [2]:
llama3 = Ollama(model="llama3", base_url='http://localhost:11434', temperature=0.0, request_timeout=600)

In [3]:
def load(filePath: str) -> str:
    """Load the Formula DSL code from the file path"""
    code = f.load(filePath)
    return code

load_tool = FunctionTool.from_defaults(fn=load)

In [4]:
def solve(partial_model:str, domain: str):
    """Try to solve the partial model based on the domain constraints"""
    """You can get the name of the partial model and the domain from the loaded file"""
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.solve(partial_model,"1",f"{domain}.conforms")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

solve_tool = FunctionTool.from_defaults(fn=solve)

In [5]:
def extract(task_id: str):
    """Extract the solve result from the task with task_id.
    If the partial model is solvable, will return the solution.
    Else if the partial model is unsolvable, will return the least unsatisfied core conditions.
    The task_id is a string representing an integer which is incremented by 1 each time we run the solve command."""
    
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.extract(task_id, "0", "0")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

extract_result_tool = FunctionTool.from_defaults(fn=extract)

In [6]:
react_agent = ReActAgent.from_tools(tools=[load_tool, solve_tool, extract_result_tool], llm=llama3, verbose=True)

In [7]:
react_system_header_str = """\
You are a helpful AI agent designed to suggest repairs to unsolvable models written in a DSL called Formula.
## Work Flow
Your typical workflow is to:
1. Load a Formula script from a user-provided file path;
2. Solve the partial model in that script;
3. Extract the solve result:
    - If the model is solvable, return the solution and your task is done.
    - If the model is unsolvable:
        1. explain why it is unsolvable
        2. suggest repairs to either the partial model or the domain constraints based on the conflicting constraints.
        3. save the modified script to a new file. Do the load, solve, extract again to see if it is solvable now.

Note:
    If the partial model is unsolvable and you need to repair it, give back the complete modified script and save it to a new file in the same folder as the original script.
    You shouldn't give back incomplete Formula script or write any code that is not syntaxically correct in Formula.
    
## Tools
You have access to four tools. You are responsible for using the tools in a correct sequence to complete the task at hand. This may require breaking the task into subtasks and using different tools to complete each subtask.

You have access to the following tools:
{tool_desc} 

## Output Format
You should choose the appropriate format based on the task at hand.

### If you need to use a tool:
```
Thought: I need to use a tool to help me answer the question.
Action: tool name (one of {tool_names})
Action Input: the input to the tool, in a JSON format representing the kwargs (e.g. {{"input": "hello world", "num_beams": 5}})
```

The response would be the tool's output in the following format:
```
Observation: tool response
```
### If you do not need to call any tool and can answer the question using you NLP ability:
```
Thought: I do not need to use any tool; Instead, I will use my NLP ability to do reasoning based on the information at hand.
Answer: [your answer here]
```

Please ensure that your responses are clear and concise, following the structure provided above.

## Current Conversation
Below is the current conversation consisting of interleaving human and assistant messages.

"""

In [8]:
react_system_prompt = PromptTemplate(react_system_header_str)
react_agent.update_prompts({"agent_worker:system_prompt": react_system_prompt})

In [9]:
prompt = """
Load the file from file path './data/MappingExample.4ml'.
Then solve the partial model call the extract tool.
If the partial model is unsolvable, the extract tool will return the unsat core terms.
Use your own natural language processing ability to explain why it is unsolvable based on the file content and the unsatisfiable core conditions.
You shouldn't need any tools for the explanation step!
Lastly, suggest a fix to the invalidUtilization constraint to solve the conflict and make the partial model solvable.
You shouldn't need any tools for the fixing step!
"""

In [10]:
response = react_agent.chat(prompt)

print(str(response))

> Running step 2e75c741-8cf1-499b-acbb-d97b4984c514. Step input: 
Load the file from file path './data/MappingExample.4ml'.
Then solve the partial model call the extract tool.
If the partial model is unsolvable, the extract tool will return the unsat core terms.
Use your own natural language processing ability to explain why it is unsolvable based on the file content and the unsatisfiable core conditions.
You shouldn't need any tools for the explanation step!
Lastly, suggest a fix to the invalidUtilization constraint to solve the conflict and make the partial model solvable.
You shouldn't need any tools for the fixing step!

Thought: I need to use a tool to help me answer the question.
Action: load
Action Input: {'filePath': './data/MappingExample.4ml'}
(Compiled) MappingExample.4ml
0.61s.
Observation: domain Mapping
{
  Component ::= new (id: Integer, utilization: Real).
  Processor ::= new (id: Integer).
  Mapping   ::= new (c: Component, p: Processor).

  // The utilization must be 